In [7]:
# Loading data fully for the experiments

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# ============================================================
# Paths
# ============================================================

RAW_DATA_DIR = Path("../data/raw")

TRAIN_PATH = RAW_DATA_DIR / "train_FD001.txt"
TEST_PATH = RAW_DATA_DIR / "test_FD001.txt"
TEST_RUL_PATH = RAW_DATA_DIR / "RUL_FD001.txt"


# ============================================================
# Column names
# ============================================================

column_names = (
    ["engine_id", "cycle"]
    + ["setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)


# ============================================================
# Load raw data
# ============================================================

train_full = pd.read_csv(
    TRAIN_PATH,
    sep=r"\s+",
    header=None,
    names=column_names,
)

test_df = pd.read_csv(
    TEST_PATH,
    sep=r"\s+",
    header=None,
    names=column_names,
)

rul_test = pd.read_csv(
    TEST_RUL_PATH,
    sep=r"\s+",
    header=None,
    names=["rul"],
)


# ============================================================
# Create RUL target for training data
# ============================================================

engine_lifetimes = (
    train_full
    .groupby("engine_id")["cycle"]
    .max()
)

train_full["max_cycle"] = (
    train_full["engine_id"]
    .map(engine_lifetimes)
)

train_full["rul"] = (
    train_full["max_cycle"]
    - train_full["cycle"]
)


# ============================================================
# Capped RUL target
# ============================================================

RUL_CAP = 125

train_full["rul_capped"] = (
    train_full["rul"]
    .clip(upper=RUL_CAP)
)


# ============================================================
# Remove constant columns
# ============================================================

constant_columns = (
    train_full
    .nunique()
    .loc[lambda x: x == 1]
    .index
    .tolist()
)

train_full = (
    train_full
    .drop(columns=constant_columns)
    .copy()
)

test_df = (
    test_df
    .drop(columns=constant_columns)
    .copy()
)


# ============================================================
# Same engine-level train / validation split
# ============================================================

engine_ids = train_full["engine_id"].unique()

train_engines, val_engines = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42,
)

train_df = train_full[
    train_full["engine_id"].isin(train_engines)
].copy()

val_df = train_full[
    train_full["engine_id"].isin(val_engines)
].copy()


# ============================================================
# Sequence feature columns
# ============================================================

sequence_feature_columns = [
    col
    for col in train_df.columns
    if col not in [
        "engine_id",
        "max_cycle",
        "rul",
        "rul_capped",
    ]
]


print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print()
print("Train engines:", len(train_engines))
print("Validation engines:", len(val_engines))

print()
print("Sequence features:")
print(sequence_feature_columns)

print()
print("Constant columns removed:")
print(constant_columns)

Train shape: (16561, 22)
Validation shape: (4070, 22)
Test shape: (13096, 19)

Train engines: 80
Validation engines: 20

Sequence features:
['cycle', 'setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

Constant columns removed:
['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


,engine_id,cycle,setting_1,setting_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,max_cycle,rul,rul_capped
192,2,1,-0.0018,0.0006,641.89,1583.84,1391.28,21.60,554.53,2388.01,...,522.33,2388.06,8137.72,8.3905,391,38.94,23.4585,287,286,125
193,2,2,0.0043,-0.0003,641.82,1587.05,1393.13,21.61,554.77,2387.98,...,522.70,2387.98,8131.09,8.4167,392,39.06,23.4085,287,285,125
194,2,3,0.0018,0.0003,641.55,1588.32,1398.96,21.60,555.14,2388.04,...,522.58,2387.99,8140.58,8.3802,391,39.11,23.4250,287,284,125
195,2,4,0.0035,-0.0004,641.68,1584.15,1396.08,21.61,554.25,2387.98,...,522.49,2387.93,8140.44,8.4018,391,39.13,23.5027,287,283,125
196,2,5,0.0005,0.0004,641.73,1579.03,1402.52,21.60,555.12,2388.03,...,522.27,2387.94,8136.67,8.3867,390,39.18,23.4234,287,282,125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,643.49,1597.98,1428.63,21.61,551.43,2388.19,...,519.49,2388.26,8137.60,8.4956,397,38.49,22.9735,200,4,4
20627,100,197,-0.0016,-0.0005,643.54,1604.50,1433.58,21.61,550.86,2388.23,...,519.68,2388.22,8136.50,8.5139,395,38.30,23.1594,200,3,3
20628,100,198,0.0004,0.0000,643.42,1602.46,1428.18,21.61,550.94,2388.24,...,520.01,2388.24,8141.05,8.5646,398,38.44,22.9333,200,2,2
20629,100,199,-0.0011,0.0003,643.23,1605.26,1426.53,21.61,550.68,2388.25,...,519.67,2388.23,8139.29,8.5389,395,38.29,23.0640,200,1,1


In [3]:
train_engines

array([ 56,  89,  27,  43,  70,  16,  41,  97,  10,  73,  12,  48,  86,
        29,  94,   6,  67,  66,  36,  17,  50,  35,   8,  96,  28,  20,
        82,  26,  63,  14,  25,   4,  18,  39,   9,  79,   7,  65,  37,
        90,  57, 100,  55,  44,  51,  68,  47,  69,  62,  98,  80,  42,
        59,  49,  99,  58,  76,  33,  95,  60,  64,  85,  38,  30,   2,
        53,  22,   3,  24,  88,  92,  75,  87,  83,  21,  61,  72,  15,
        93,  52])

In [5]:
import copy
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. Create overlapping sequences
# ============================================================

def create_sequences(
    df,
    feature_columns,
    target_column="rul_capped",
    window=30,
):
    """
    Convert engine trajectories into overlapping fixed-length sequences.

    Example with window=30:

        cycles 1-30  -> target at cycle 30
        cycles 2-31  -> target at cycle 31
        cycles 3-32  -> target at cycle 32
        ...

    Returns
    -------
    X : np.ndarray
        Shape:
        (number_of_sequences, window, number_of_features)

    y : np.ndarray
        Shape:
        (number_of_sequences,)
    """
    X = []
    y = []

    for engine_id, engine_data in df.groupby("engine_id"):

        engine_data = engine_data.sort_values("cycle")

        features = (
            engine_data[feature_columns]
            .to_numpy()
        )

        targets = (
            engine_data[target_column]
            .to_numpy()
        )

        for end_idx in range(
            window - 1,
            len(engine_data),
        ):

            start_idx = (
                end_idx
                - window
                + 1
            )

            sequence = features[
                start_idx:end_idx + 1
            ]

            target = targets[end_idx]

            X.append(sequence)
            y.append(target)

    return (
        np.array(X),
        np.array(y),
    )


# ============================================================
# 2. Scale sequence data
# ============================================================

def scale_sequences(
    X_train,
    X_val,
):
    """
    Standardize each feature using training data only.

    The sequence tensors are temporarily reshaped from:

        (samples, timesteps, features)

    into:

        (samples * timesteps, features)

    so StandardScaler can scale each feature independently.

    Returns
    -------
    X_train_scaled
    X_val_scaled
    scaler
    """
    scaler = StandardScaler()

    n_train, timesteps, n_features = (
        X_train.shape
    )

    n_val = X_val.shape[0]

    # Flatten temporal dimension temporarily.
    X_train_2d = X_train.reshape(
        -1,
        n_features,
    )

    X_val_2d = X_val.reshape(
        -1,
        n_features,
    )

    # Fit only on training data.
    X_train_scaled_2d = (
        scaler.fit_transform(
            X_train_2d
        )
    )

    X_val_scaled_2d = (
        scaler.transform(
            X_val_2d
        )
    )

    # Restore sequence shape.
    X_train_scaled = (
        X_train_scaled_2d.reshape(
            n_train,
            timesteps,
            n_features,
        )
    )

    X_val_scaled = (
        X_val_scaled_2d.reshape(
            n_val,
            timesteps,
            n_features,
        )
    )

    return (
        X_train_scaled,
        X_val_scaled,
        scaler,
    )


# ============================================================
# 3. LSTM model
# ============================================================

class RULPredictor(nn.Module):
    """
    LSTM-based Remaining Useful Life predictor.
    """

    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=1,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.output_layer = nn.Linear(
            hidden_size,
            1,
        )

    def forward(self, x):
        """
        Predict one RUL value per input sequence.
        """
        lstm_output, _ = self.lstm(x)

        # Representation after final timestep.
        final_hidden = (
            lstm_output[:, -1, :]
        )

        prediction = (
            self.output_layer(
                final_hidden
            )
        )

        return prediction.squeeze(1)


# ============================================================
# 4. Asymmetric loss
# ============================================================

def asymmetric_mse(
    predictions,
    targets,
    overprediction_weight=2.0,
):
    """
    Penalize RUL overprediction more heavily.

    Positive error means:

        prediction > target

    which means the model thinks the engine has more
    remaining life than it actually has.
    """
    errors = (
        predictions
        - targets
    )

    weights = torch.where(
        errors > 0,
        torch.full_like(
            errors,
            overprediction_weight,
        ),
        torch.ones_like(errors),
    )

    return torch.mean(
        weights
        * errors.pow(2)
    )


# ============================================================
# 5. Create DataLoaders
# ============================================================

def create_loaders(
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size=64,
):
    """
    Convert NumPy arrays to PyTorch tensors and DataLoaders.
    """
    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32,
    )

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32,
    )

    y_val_tensor = torch.tensor(
        y_val,
        dtype=torch.float32,
    )

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor,
    )

    val_dataset = TensorDataset(
        X_val_tensor,
        y_val_tensor,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return (
        train_loader,
        val_loader,
        X_val_tensor,
        y_val_tensor,
    )


# ============================================================
# 6. Train model and keep best validation checkpoint
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    epochs=30,
    learning_rate=0.001,
    overprediction_weight=2.0,
):
    """
    Train an LSTM using asymmetric MSE.

    The best model is chosen using validation loss.
    """
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    best_val_loss = float("inf")
    best_model_state = None
    best_epoch = None

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(epochs):

        # -------------------------
        # Training
        # -------------------------

        model.train()

        train_loss = 0.0

        for X_batch, y_batch in train_loader:

            optimizer.zero_grad()

            predictions = model(
                X_batch
            )

            loss = asymmetric_mse(
                predictions,
                y_batch,
                overprediction_weight=(
                    overprediction_weight
                ),
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item()
                * X_batch.size(0)
            )

        train_loss /= len(
            train_loader.dataset
        )

        # -------------------------
        # Validation
        # -------------------------

        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                predictions = model(
                    X_batch
                )

                loss = asymmetric_mse(
                    predictions,
                    y_batch,
                    overprediction_weight=(
                        overprediction_weight
                    ),
                )

                val_loss += (
                    loss.item()
                    * X_batch.size(0)
                )

        val_loss /= len(
            val_loader.dataset
        )

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        # Save best checkpoint.
        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch + 1

            best_model_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss:.2f} | "
            f"Val Loss: {val_loss:.2f}"
        )

    # Restore best checkpoint.
    model.load_state_dict(
        best_model_state
    )

    return (
        model,
        history,
        best_epoch,
        best_val_loss,
    )


# ============================================================
# 7. Evaluate model
# ============================================================

def evaluate_model(
    model,
    X_tensor,
    y_true,
):
    """
    Calculate predictions, MAE, MSE, and RMSE.
    """
    model.eval()

    with torch.no_grad():

        predictions = (
            model(X_tensor)
            .cpu()
            .numpy()
        )

    mae = mean_absolute_error(
        y_true,
        predictions,
    )

    mse = mean_squared_error(
        y_true,
        predictions,
    )

    rmse = np.sqrt(mse)

    return {
        "predictions": predictions,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
    }

In [6]:
# ============================================================
# Experiment A: Full 30-cycle sequence
# ============================================================

WINDOW = 30

# Create overlapping sequences.
X_train_full, y_train_full = create_sequences(
    train_df,
    feature_columns=sequence_feature_columns,
    target_column="rul_capped",
    window=WINDOW,
)

X_val_full, y_val_full = create_sequences(
    val_df,
    feature_columns=sequence_feature_columns,
    target_column="rul_capped",
    window=WINDOW,
)

print("Before scaling")
print("X_train:", X_train_full.shape)
print("X_val:", X_val_full.shape)


# Scale using training data only.
X_train_full_scaled, X_val_full_scaled, full_scaler = (
    scale_sequences(
        X_train_full,
        X_val_full,
    )
)


# Create PyTorch loaders.
(
    train_loader_full,
    val_loader_full,
    X_val_full_tensor,
    y_val_full_tensor,
) = create_loaders(
    X_train_full_scaled,
    y_train_full,
    X_val_full_scaled,
    y_val_full,
    batch_size=64,
)


# Create a fresh model.
model_full = RULPredictor(
    input_size=X_train_full_scaled.shape[2],
    hidden_size=64,
    num_layers=1,
)


# Train.
(
    model_full,
    history_full,
    best_epoch_full,
    best_val_loss_full,
) = train_model(
    model_full,
    train_loader_full,
    val_loader_full,
    epochs=30,
    learning_rate=0.001,
    overprediction_weight=2.0,
)


# Evaluate the best checkpoint.
results_full = evaluate_model(
    model_full,
    X_val_full_tensor,
    y_val_full,
)


print()
print("Full sequence baseline")
print(f"Best epoch: {best_epoch_full}")
print(f"Validation MAE:  {results_full['mae']:.2f}")
print(f"Validation MSE:  {results_full['mse']:.2f}")
print(f"Validation RMSE: {results_full['rmse']:.2f}")

Before scaling
X_train: (14241, 30, 18)
X_val: (3490, 30, 18)
Epoch 01 | Train Loss: 6721.76 | Val Loss: 5412.67
Epoch 02 | Train Loss: 4719.34 | Val Loss: 3905.96
Epoch 03 | Train Loss: 3429.12 | Val Loss: 2838.64
Epoch 04 | Train Loss: 2508.46 | Val Loss: 2082.76
Epoch 05 | Train Loss: 1851.55 | Val Loss: 1554.33
Epoch 06 | Train Loss: 1373.25 | Val Loss: 1131.68
Epoch 07 | Train Loss: 1022.22 | Val Loss: 842.22
Epoch 08 | Train Loss: 767.03 | Val Loss: 640.03
Epoch 09 | Train Loss: 585.29 | Val Loss: 488.12
Epoch 10 | Train Loss: 453.53 | Val Loss: 394.25
Epoch 11 | Train Loss: 362.89 | Val Loss: 333.64
Epoch 12 | Train Loss: 294.41 | Val Loss: 286.35
Epoch 13 | Train Loss: 251.88 | Val Loss: 263.40
Epoch 14 | Train Loss: 212.53 | Val Loss: 285.98
Epoch 15 | Train Loss: 189.31 | Val Loss: 250.65
Epoch 16 | Train Loss: 159.96 | Val Loss: 259.23
Epoch 17 | Train Loss: 144.98 | Val Loss: 253.85
Epoch 18 | Train Loss: 134.54 | Val Loss: 283.20
Epoch 19 | Train Loss: 111.52 | Val Loss: 3

### Is the final timestamp(latest snapshot of senesors) enough or we really need 30-cycle history?

In [10]:
# ============================================================
# Experiment B: Final timestep only
# ============================================================

# Take only the last timestep from each 30-cycle sequence.
# Keep a sequence dimension of length 1 so the same LSTM model can be used.
X_train_last = X_train_full_scaled[:, -1:, :]
X_val_last = X_val_full_scaled[:, -1:, :]

print("Experiment B: Final timestep only")
print("X_train:", X_train_last.shape)
print("X_val:", X_val_last.shape)


# Create PyTorch loaders.
(
    train_loader_last,
    val_loader_last,
    X_val_last_tensor,
    y_val_last_tensor,
) = create_loaders(
    X_train_last,
    y_train_full,
    X_val_last,
    y_val_full,
    batch_size=64,
)


# Fresh model.
model_last = RULPredictor(
    input_size=X_train_last.shape[2],
    hidden_size=64,
    num_layers=1,
)


# Train.
(
    model_last,
    history_last,
    best_epoch_last,
    best_val_loss_last,
) = train_model(
    model_last,
    train_loader_last,
    val_loader_last,
    epochs=60,
    learning_rate=0.001,
    overprediction_weight=2.0,
)


# Evaluate.
results_last = evaluate_model(
    model_last,
    X_val_last_tensor,
    y_val_full,
)


print()
print("Experiment B: Final timestep only")
print(f"Best epoch: {best_epoch_last}")
print(
    f"Experiment B - Final timestep only - "
    f"Validation MAE: {results_last['mae']:.2f}"
)
print(
    f"Experiment B - Final timestep only - "
    f"Validation MSE: {results_last['mse']:.2f}"
)
print(
    f"Experiment B - Final timestep only - "
    f"Validation RMSE: {results_last['rmse']:.2f}"
)

Experiment B: Final timestep only
X_train: (14241, 1, 18)
X_val: (3490, 1, 18)
Epoch 01 | Train Loss: 7760.49 | Val Loss: 6810.40
Epoch 02 | Train Loss: 6141.14 | Val Loss: 5258.04
Epoch 03 | Train Loss: 4590.46 | Val Loss: 3707.45
Epoch 04 | Train Loss: 3205.01 | Val Loss: 2530.64
Epoch 05 | Train Loss: 2201.79 | Val Loss: 1743.23
Epoch 06 | Train Loss: 1552.29 | Val Loss: 1264.55
Epoch 07 | Train Loss: 1169.53 | Val Loss: 971.44
Epoch 08 | Train Loss: 934.08 | Val Loss: 782.04
Epoch 09 | Train Loss: 784.79 | Val Loss: 661.23
Epoch 10 | Train Loss: 690.43 | Val Loss: 583.00
Epoch 11 | Train Loss: 631.39 | Val Loss: 531.65
Epoch 12 | Train Loss: 594.45 | Val Loss: 498.39
Epoch 13 | Train Loss: 570.87 | Val Loss: 474.55
Epoch 14 | Train Loss: 555.30 | Val Loss: 457.15
Epoch 15 | Train Loss: 543.79 | Val Loss: 442.64
Epoch 16 | Train Loss: 535.08 | Val Loss: 430.82
Epoch 17 | Train Loss: 527.38 | Val Loss: 420.21
Epoch 18 | Train Loss: 520.46 | Val Loss: 410.49
Epoch 19 | Train Loss: 513